# Databricks Group Audit Tool

Audit Databricks group membership and Unity Catalog permissions across all workspaces.

## Modes

| Mode | Widget to set | Question answered |
|------|--------------|------------------|
| **Group audit** | `target_group` | What does this group access? Who has redundant personal grants? |
| **Principal audit** | `principal_identifier` | What can this user / SP / group access across the entire account? |

Set one widget above, then click **Run all**.

## Features
- Recursive group membership with IdP source tagging (`external` / `internal`)
- Multi-workspace catalog, schema, and table permission scanning
- Workspace-level object permission scanning — 13 types: jobs, clusters, cluster policies, DLT pipelines, SQL warehouses, SQL queries, SQL alerts, Lakeview dashboards, Genie spaces, MLflow experiments, registered models, serving endpoints, Databricks Apps
- Redundancy detection with copy-paste REVOKE SQL
- Privilege escalation detection (`ALL_PRIVILEGES` / `MANAGE`)
- Stale grant detection via `system.access.audit`
- Workspace-local (legacy pre-UC) group detection
- CSV export for Excel / SIEM ingestion
- JSON snapshots + diff for SOC 2 / ISO 27001 compliance evidence


In [ ]:
# Run once per cluster restart to install / upgrade the package.
# Adjust the path to match your repo location in the workspace.
%pip install -q "/Workspace/Users/your.name@company.com/databricks-group-audit-tool[sdk]"
dbutils.library.restartPython()

In [ ]:
import io, json, os
from datetime import datetime
from typing import Dict, List

from databricks_group_audit import (
    # Clients
    create_client,
    # Core modules
    GroupMembershipResolver, WorkspaceDiscovery,
    CatalogPermissionScanner, SchemaPermissionScanner, TablePermissionScanner,
    RedundancyDetector, RevokeScriptGenerator,
    WorkspaceObjectScanner,
    PrincipalAuditor, PermissionElevator,
    detect_escalations, StaleGrantChecker, LocalGroupChecker,
    # CSV + snapshot
    write_group_audit_csv, write_principal_audit_csv,
    build_group_snapshot, build_principal_snapshot,
    save_snapshot, load_snapshot, diff_snapshots,
    # Models
    MemberType, PrincipalSource,
    GroupMember, GroupNode, WorkspaceInfo,
    GrantSource, CatalogGrant, SchemaGrant, TableGrant, WorkspaceObjectGrant,
    RedundancyLevel, RedundancyResult,
    WORKSPACE_DOMAIN_MAP, classify_grant, build_member_lookups,
)
import databricks_group_audit as _pkg
print(f"\u2705 databricks-group-audit v{_pkg.__version__} loaded")


In [ ]:
# ============================================================================
# CONFIGURATION  -  fill widgets in the header bar or set values directly here
# ============================================================================
_w = {}
try:
    dbutils.widgets.text("secret_scope",         "",                                           "Secret scope (overrides plain-text creds)")
    dbutils.widgets.text("client_id",            os.getenv("DATABRICKS_CLIENT_ID", ""),       "SP Client ID")
    dbutils.widgets.text("client_secret",        os.getenv("DATABRICKS_CLIENT_SECRET", ""),   "SP Client Secret")
    dbutils.widgets.text("account_id",           os.getenv("DATABRICKS_ACCOUNT_ID", ""),      "Account ID")
    dbutils.widgets.dropdown("cloud",            "azure", ["azure", "aws", "gcp"],             "Cloud")
    dbutils.widgets.text("target_group",         "",   "Group to audit")
    dbutils.widgets.text("principal_identifier", "",   "Principal to audit (email / SP / group)")
    dbutils.widgets.text("workspace_urls",       "",   "Workspace URLs (comma-sep; blank = all)")
    dbutils.widgets.dropdown("scan_schemas",     "false", ["true", "false"], "Scan schemas")
    dbutils.widgets.dropdown("scan_tables",      "false", ["true", "false"], "Scan tables")
    dbutils.widgets.text("workers",              "8",  "Parallel workers (default: 8)")
    dbutils.widgets.dropdown("auto_elevate",     "false", ["true", "false"], "Auto-elevate permissions")
    dbutils.widgets.dropdown("dry_run_elevation","false", ["true", "false"], "Dry-run elevation only")
    dbutils.widgets.dropdown("escalation_check", "false", ["true", "false"], "Escalation check (principal)")
    dbutils.widgets.text("stale_days",           "0",  "Stale-grant threshold days (0 = off)")
    dbutils.widgets.text("sql_warehouse_id",     "",   "SQL warehouse ID (stale check)")
    dbutils.widgets.text("sql_workspace_url",    "",   "Workspace URL for stale check")
    dbutils.widgets.dropdown("check_local_groups","false",["true","false"],  "Check local groups")
    dbutils.widgets.text("save_snapshot",        "",   "Save snapshot to path (optional)")
    dbutils.widgets.text("baseline_snapshot",    "",   "Baseline snapshot path (diff, optional)")
    dbutils.widgets.text("export_delta_path",    "",   "Delta export path (optional)")
    dbutils.widgets.dropdown("scan_workspace_objects","false",["true","false"], "Scan workspace objects")
    dbutils.widgets.text("workspace_object_types", "",   "Object types (blank=all; jobs,clusters,...)")
    def _get(k): return dbutils.widgets.get(k).strip()
except Exception:
    def _get(k): return _w.get(k, "")

# ---------------------------------------------------------------------------
# Secret scope — if provided, client_id / client_secret / account_id are
# read from dbutils.secrets (keys must match those names in the scope).
# Falls back to widget values or environment variables when scope is absent
# or a key does not exist in the scope.
# ---------------------------------------------------------------------------
SECRET_SCOPE = _get("secret_scope")

def _secret(key, fallback):
    if SECRET_SCOPE:
        try:
            return dbutils.secrets.get(scope=SECRET_SCOPE, key=key)
        except Exception:
            pass
    return fallback

CLIENT_ID          = _secret("client_id",     _get("client_id")     or os.getenv("DATABRICKS_CLIENT_ID", ""))
CLIENT_SECRET      = _secret("client_secret", _get("client_secret") or os.getenv("DATABRICKS_CLIENT_SECRET", ""))
ACCOUNT_ID         = _secret("account_id",    _get("account_id")    or os.getenv("DATABRICKS_ACCOUNT_ID", ""))
CLOUD              = _get("cloud")               or "azure"
TARGET_GROUP       = _get("target_group")
PRINCIPAL          = _get("principal_identifier")
WORKSPACE_URLS     = _get("workspace_urls")
SCAN_SCHEMAS       = _get("scan_schemas")        == "true"
SCAN_TABLES        = _get("scan_tables")         == "true"
WORKERS            = int(_get("workers") or 8)
AUTO_ELEVATE       = _get("auto_elevate")        == "true"
DRY_RUN_ELEV       = _get("dry_run_elevation")   == "true"
ESCALATION_CHECK   = _get("escalation_check")    == "true"
STALE_DAYS         = int(_get("stale_days") or 0)
SQL_WAREHOUSE_ID   = _get("sql_warehouse_id")
SQL_WORKSPACE_URL  = _get("sql_workspace_url")
CHECK_LOCAL_GROUPS = _get("check_local_groups")  == "true"
SAVE_SNAPSHOT      = _get("save_snapshot")
BASELINE_SNAPSHOT  = _get("baseline_snapshot")
EXPORT_DELTA_PATH         = _get("export_delta_path")
SCAN_WORKSPACE_OBJECTS    = _get("scan_workspace_objects") == "true"
WORKSPACE_OBJECT_TYPES    = _get("workspace_object_types")

if not CLIENT_ID or not CLIENT_SECRET or not ACCOUNT_ID:
    print("⚠️  Set CLIENT_ID, CLIENT_SECRET, and ACCOUNT_ID before running audit cells.")
else:
    src = f"secret scope '{SECRET_SCOPE}'" if SECRET_SCOPE else "widget / env"
    mode = "group" if TARGET_GROUP else ("principal" if PRINCIPAL else "none")
    print(f"✅ Config loaded  cloud={CLOUD}  mode={mode}  workers={WORKERS}  target='{TARGET_GROUP or PRINCIPAL}'  creds from: {src}")

In [ ]:
# ============================================================================
# BUILD CLIENTS AND SERVICES
# ============================================================================
client            = create_client(cloud=CLOUD, client_id=CLIENT_ID,
                                   client_secret=CLIENT_SECRET, account_id=ACCOUNT_ID)
group_resolver    = GroupMembershipResolver(client)
ws_discovery      = WorkspaceDiscovery(client, cloud_provider=CLOUD)
cat_scanner       = CatalogPermissionScanner(client, group_resolver)
sch_scanner       = SchemaPermissionScanner(client)
tbl_scanner       = TablePermissionScanner(client)
redundancy_det    = RedundancyDetector()
obj_scanner       = WorkspaceObjectScanner(client, group_resolver)
# PrincipalAuditor shares the resolver so get_group_membership_map() is cached across both modes
pa_auditor        = PrincipalAuditor(client, workspace_discovery=ws_discovery,
                                     cloud_provider=CLOUD, group_resolver=group_resolver)

print("\u2705 Clients and services initialized")


In [ ]:
from collections import Counter
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, BooleanType

class AuditResultBuilder:
    """Builds Spark DataFrames from audit results for display and Delta export."""

    def __init__(self, spark):
        self.spark = spark

    def _empty(self, fields):
        return self.spark.createDataFrame([], StructType([StructField(f, StringType(), True) for f in fields]))

    def build_permissions_df(self, grants):
        rows = [Row(workspace=g.workspace_name, securable_type="CATALOG",
                    securable_name=g.catalog_name, principal=g.principal,
                    principal_type=g.principal_type, privileges=", ".join(g.privileges),
                    grant_source=g.grant_source.value, inherited_from=g.inherited_from or "")
                for g in grants]
        return self.spark.createDataFrame(rows) if rows else self._empty(
            ["workspace","securable_type","securable_name","principal",
             "principal_type","privileges","grant_source","inherited_from"])

    def build_schema_grants_df(self, grants):
        rows = [Row(workspace=g.workspace_name, securable_name=f"{g.catalog_name}.{g.schema_name}",
                    principal=g.principal, principal_type=g.principal_type,
                    privileges=", ".join(g.privileges), grant_source=g.grant_source.value,
                    inherited_from=g.inherited_from or "")
                for g in grants]
        return self.spark.createDataFrame(rows) if rows else self._empty(
            ["workspace","securable_name","principal","principal_type",
             "privileges","grant_source","inherited_from"])

    def build_table_grants_df(self, grants):
        rows = [Row(workspace=g.workspace_name, full_name=g.full_name, table_type=g.table_type,
                    principal=g.principal, principal_type=g.principal_type,
                    privileges=", ".join(g.privileges), grant_source=g.grant_source.value)
                for g in grants]
        return self.spark.createDataFrame(rows) if rows else self._empty(
            ["workspace","full_name","table_type","principal",
             "principal_type","privileges","grant_source"])

    def build_membership_df(self, members):
        rows = []
        for u in members.get("users", []):
            rows.append(Row(id=u.id, display_name=u.display_name, type="USER",
                            email=u.email or "", source=u.source.value,
                            parent_groups=" -> ".join(u.parent_groups)))
        for sp in members.get("service_principals", []):
            rows.append(Row(id=sp.id, display_name=sp.display_name, type="SERVICE_PRINCIPAL",
                            email="", source=sp.source.value,
                            parent_groups=" -> ".join(sp.parent_groups)))
        return self.spark.createDataFrame(rows) if rows else self._empty(
            ["id","display_name","type","email","source","parent_groups"])

    def build_redundancy_df(self, results):
        rows = [Row(catalog=r.catalog_name, principal=r.principal, principal_type=r.principal_type,
                    member_privileges=", ".join(r.member_privileges),
                    group_effective=", ".join(sorted(r.group_effective_privileges)),
                    redundant_privileges=", ".join(r.redundant_privileges),
                    redundancy_level=r.redundancy_level.value, recommendation=r.recommendation)
                for r in results]
        return self.spark.createDataFrame(rows) if rows else self._empty(
            ["catalog","principal","principal_type","member_privileges",
             "group_effective","redundant_privileges","redundancy_level","recommendation"])

    def build_top_members_df(self, grants, redundancy):
        """Rank members by number of personal (member-direct) catalog grants."""
        grant_counts = Counter(
            g.principal for g in grants if g.grant_source == GrantSource.MEMBER_DIRECT
        )
        redundancy_map = {r.principal: r.redundancy_level.value for r in redundancy}
        rows = [
            Row(rank=str(i), principal=principal, personal_grants=str(count),
                redundancy=redundancy_map.get(principal, "None"))
            for i, (principal, count) in enumerate(grant_counts.most_common(), 1)
        ]
        return self.spark.createDataFrame(rows) if rows else self._empty(
            ["rank","principal","personal_grants","redundancy"])

    def build_principal_groups_df(self, groups):
        rows = [Row(group_name=g.group_name, is_direct=g.is_direct,
                    source=g.source.value, path=" -> ".join(g.path))
                for g in groups]
        return self.spark.createDataFrame(rows) if rows else self._empty(
            ["group_name","is_direct","source","path"])

    def build_principal_ws_df(self, roles):
        rows = [Row(workspace=r.workspace_name, permission_level=r.permission_level, via_group=r.via_group)
                for r in roles]
        return self.spark.createDataFrame(rows) if rows else self._empty(
            ["workspace","permission_level","via_group"])

    def build_principal_perms_df(self, perms):
        rows = [Row(securable_type=p.securable_type, securable_name=p.securable_name,
                    privileges=", ".join(p.privileges), via_group=p.via_group,
                    workspace=p.workspace_name)
                for p in perms]
        return self.spark.createDataFrame(rows) if rows else self._empty(
            ["securable_type","securable_name","privileges","via_group","workspace"])

    def build_stale_df(self, findings):
        rows = [Row(principal=f.principal, principal_type=f.principal_type,
                    catalog=f.catalog_name, privileges=", ".join(f.privileges),
                    last_access=f.last_access or "never", stale_days=str(f.stale_days),
                    workspace=f.workspace_name)
                for f in findings]
        return self.spark.createDataFrame(rows) if rows else self._empty(
            ["principal","principal_type","catalog","privileges","last_access","stale_days","workspace"])

    def build_escalation_df(self, findings):
        rows = [Row(principal=f.principal_name, privilege=f.privilege,
                    securable_type=f.securable_type, securable_name=f.securable_name,
                    via_group=f.via_group, is_transitive=str(f.is_transitive),
                    workspace=f.workspace_name)
                for f in findings]
        return self.spark.createDataFrame(rows) if rows else self._empty(
            ["principal","privilege","securable_type","securable_name",
             "via_group","is_transitive","workspace"])

    def build_local_groups_df(self, findings):
        rows = [Row(group_name=f.group_name, workspace=f.workspace_name,
                    member_count=str(f.member_count))
                for f in findings]
        return self.spark.createDataFrame(rows) if rows else self._empty(
            ["group_name","workspace","member_count"])


    def build_workspace_objects_df(self, grants):
        rows = [Row(object_type=g.object_type, object_name=g.object_name or g.object_id,
                    workspace=g.workspace_name, principal=g.principal,
                    principal_type=g.principal_type, permission_level=g.permission_level,
                    grant_source=g.grant_source.value, inherited_from=g.inherited_from or "")
                for g in grants]
        return self.spark.createDataFrame(rows) if rows else self._empty(
            ["object_type","object_name","workspace","principal",
             "principal_type","permission_level","grant_source","inherited_from"])

result_builder = AuditResultBuilder(spark)
print("✅ AuditResultBuilder initialized")

---
## Group Audit

Resolve the group tree, scan all workspaces, detect redundancy, and populate result DataFrames.
Set `target_group` in the configuration above.


In [ ]:
# ============================================================================
# GROUP AUDIT - main execution
# ============================================================================
if not TARGET_GROUP:
    print("⏩  Skipped - set target_group widget to run group audit")
else:
    print("\n" + "="*70)
    print("  GROUP AUDIT:", TARGET_GROUP)
    print("="*70)

    # Step 1: Resolve group
    print("\nStep 1: Resolving group membership ...")
    _group_node = group_resolver.resolve_group(TARGET_GROUP)
    if not _group_node:
        raise ValueError(f"Group '{TARGET_GROUP}' not found")
    _members = group_resolver.get_all_members_flat(_group_node)
    _ext_users = sum(1 for u in _members["users"] if u.external_id)
    _ext_sps   = sum(1 for sp in _members["service_principals"] if sp.external_id)
    print(f"  Users:             {len(_members['users'])} ({_ext_users} IdP-synced, {len(_members['users'])-_ext_users} Databricks-managed)")
    print(f"  Service principals:{len(_members['service_principals'])} ({_ext_sps} IdP-synced, {len(_members['service_principals'])-_ext_sps} Databricks-managed)")

    # Step 2: Discover workspaces
    print("\nStep 2: Discovering workspaces ...")
    _workspaces = ws_discovery.discover(WORKSPACE_URLS)
    print(f"  Found {len(_workspaces)} workspace(s)")

    # Optional permission elevation
    import contextlib
    _elev_ctx = contextlib.nullcontext(None)
    if AUTO_ELEVATE or DRY_RUN_ELEV:
        _elevator = PermissionElevator(client, CLIENT_ID, dry_run=DRY_RUN_ELEV)
        _elevator.__enter__()
        try:
            for _ws in _workspaces:
                _elevator.ensure_workspace_admin(_ws.workspace_id, _ws.workspace_name)
        except Exception:
            _elevator.__exit__(*__import__('sys').exc_info())
            raise
        from databricks_group_audit.cli import _AlreadyEnteredContext
        _elev_ctx = _AlreadyEnteredContext(_elevator)
        print(f"  Elevation: {'DRY RUN' if DRY_RUN_ELEV else 'active'}")

    # Step 3: Scan catalog permissions
    print("\nStep 3: Scanning catalog permissions ...")
    _url_to_ws_name = {ws.workspace_url: ws.workspace_name for ws in _workspaces}
    with _elev_ctx:
        _cat_grants = cat_scanner.scan_all_workspaces(
            _workspaces, TARGET_GROUP, _group_node, _members, max_workers=WORKERS,
        )
        print(f"  Found {len(_cat_grants)} catalog grant(s)")


        # Step 4: Workspace object permissions (optional)
        _ws_obj_grants = []
        if SCAN_WORKSPACE_OBJECTS:
            print("\nStep 4: Scanning workspace object permissions ...")
            _obj_types = [t.strip() for t in WORKSPACE_OBJECT_TYPES.split(",") if t.strip()] or None
            _ws_obj_grants = obj_scanner.scan_all_workspaces(
                _workspaces, TARGET_GROUP, _group_node, _members,
                object_types=_obj_types, max_workers=WORKERS,
            )
            print(f"  Found {len(_ws_obj_grants)} workspace object grant(s)")
            print("  Note: remediation requires the Databricks permissions REST API, not SQL.")

        # Step 5: Schema scanning
        _sch_grants = []
        if SCAN_SCHEMAS or SCAN_TABLES:
            print("\nStep 5: Scanning schema permissions ...")
            _upstream = cat_scanner.get_groups_containing_target(TARGET_GROUP)
            _accessible = {(g.catalog_name, g.workspace_url) for g in _cat_grants
                           if g.grant_source in (GrantSource.DIRECT, GrantSource.UPSTREAM)}
            for _cname, _wurl in _accessible:
                _ws = WorkspaceInfo("scan", "", _url_to_ws_name.get(_wurl, ""), _wurl, CLOUD.upper(), "")
                _sch_grants.extend(sch_scanner.scan_schemas(_ws, _cname, TARGET_GROUP, _members, _upstream))
            print(f"  Found {len(_sch_grants)} schema grant(s)")

        # Step 6: Table scanning
        _tbl_grants = []
        if SCAN_TABLES:
            print("\nStep 6: Scanning table permissions ...")
            for _cname, _wurl in _accessible:
                _ws = WorkspaceInfo("scan", "", _url_to_ws_name.get(_wurl, ""), _wurl, CLOUD.upper(), "")
                for _sch in sch_scanner.get_schemas(_ws, _cname):
                    _tbl_grants.extend(tbl_scanner.scan_tables(_ws, _cname, _sch.get("name", ""),
                                                                TARGET_GROUP, _members, _upstream))
            print(f"  Found {len(_tbl_grants)} table grant(s)")

    # Step 7: Redundancy
    print("\nStep 7: Detecting redundant grants ...")
    _redundancy = redundancy_det.detect_redundancy(_cat_grants, TARGET_GROUP)
    _full    = sum(1 for r in _redundancy if r.redundancy_level == RedundancyLevel.FULL)
    _partial = sum(1 for r in _redundancy if r.redundancy_level == RedundancyLevel.PARTIAL)
    print(f"  {_full} fully redundant, {_partial} partially redundant")

    # Build DataFrames
    df_grants        = result_builder.build_permissions_df(_cat_grants)
    df_membership    = result_builder.build_membership_df(_members)
    df_redundancy    = result_builder.build_redundancy_df(_redundancy)
    df_schema_grants = result_builder.build_schema_grants_df(_sch_grants)
    df_table_grants  = result_builder.build_table_grants_df(_tbl_grants)
    df_top_members        = result_builder.build_top_members_df(_cat_grants, _redundancy)
    df_workspace_objects  = result_builder.build_workspace_objects_df(_ws_obj_grants)
    _revoke_sql      = RevokeScriptGenerator.generate(_redundancy, include_partial=True)

    print("\n✅ Group audit complete")
    print("   df_grants, df_membership, df_redundancy, df_schema_grants, df_table_grants, df_top_members, _revoke_sql")
    if SCAN_WORKSPACE_OBJECTS:
        print("   df_workspace_objects")

In [ ]:
if 'df_grants' in dir() and TARGET_GROUP:
    print("\U0001f4cb Catalog Grants")
    display(df_grants)


In [ ]:
if 'df_membership' in dir() and TARGET_GROUP:
    print("\U0001f465 Group Membership  (source: external = IdP-synced, internal = Databricks-managed)")
    display(df_membership)


In [ ]:
if 'df_redundancy' in dir() and TARGET_GROUP:
    print("\u26a0\ufe0f  Redundancy Analysis")
    display(df_redundancy)


In [ ]:
if 'df_top_members' in dir() and TARGET_GROUP:
    if df_top_members.count() > 0:
        print("🏆 Top Members by Personal Grants  (candidates for cleanup)")
        display(df_top_members)
    else:
        print("ℹ️  No member-direct grants found — no personal grants to rank")

In [ ]:
if 'df_schema_grants' in dir() and TARGET_GROUP:
    if df_schema_grants.count() > 0:
        print("\U0001f4c1 Schema Grants")
        display(df_schema_grants)
    else:
        print("\u2139\ufe0f  No schema grants (enable scan_schemas widget to scan)")

if 'df_table_grants' in dir() and TARGET_GROUP:
    if df_table_grants.count() > 0:
        print("\U0001f4dd Table / View Grants")
        display(df_table_grants)
    else:
        print("\u2139\ufe0f  No table grants (enable scan_tables widget to scan)")


In [ ]:
if '_revoke_sql' in dir() and TARGET_GROUP:
    print("\U0001f4dc REVOKE script (review carefully before executing):\n")
    print(_revoke_sql)


if 'df_workspace_objects' in dir() and TARGET_GROUP and SCAN_WORKSPACE_OBJECTS:
    if df_workspace_objects.count() > 0:
        print("🔧 Workspace Object Permissions")
        display(df_workspace_objects)
    else:
        print("ℹ️  No workspace object grants found")

---
## Principal Audit

Reverse lookup: starting from a user, service principal, or group, resolve every group
membership, workspace role, and Unity Catalog permission across the entire account.

Set `principal_identifier` in the configuration above.


In [ ]:
# ============================================================================
# PRINCIPAL AUDIT - main execution
# ============================================================================
if not PRINCIPAL:
    print("⏩  Skipped - set principal_identifier widget to run principal audit")
else:
    print("\n" + "="*70)
    print("  PRINCIPAL AUDIT:", PRINCIPAL)
    print("="*70)

    _pa = pa_auditor  # shares group_resolver cache with group audit
    _pa_workspaces = ws_discovery.discover(WORKSPACE_URLS)

    import contextlib
    _pa_elev_ctx = contextlib.nullcontext(None)
    if AUTO_ELEVATE or DRY_RUN_ELEV:
        _pa_elevator = PermissionElevator(client, CLIENT_ID, dry_run=DRY_RUN_ELEV)
        _pa_elevator.__enter__()
        try:
            for _ws in _pa_workspaces:
                _pa_elevator.ensure_workspace_admin(_ws.workspace_id, _ws.workspace_name)
        except Exception:
            _pa_elevator.__exit__(*__import__('sys').exc_info())
            raise
        from databricks_group_audit.cli import _AlreadyEnteredContext
        _pa_elev_ctx = _AlreadyEnteredContext(_pa_elevator)

    with _pa_elev_ctx:
        _pa_result = _pa.audit(
            identifier=PRINCIPAL,
            explicit_workspace_urls=WORKSPACE_URLS,
            scan_schemas=SCAN_SCHEMAS,
            scan_tables=SCAN_TABLES,
            scan_workspace_objects=SCAN_WORKSPACE_OBJECTS,
            workspace_object_types=([t.strip() for t in WORKSPACE_OBJECT_TYPES.split(",") if t.strip()] or None),
            max_workers=WORKERS,
        )

    _p_src = _pa_result.principal_source.value
    _ext_g  = sum(1 for g in _pa_result.groups if g.external_id)
    print(f"\n  Principal: {_pa_result.principal_name} ({_pa_result.principal_type}, {_p_src})")
    print(f"  Groups:    {len(_pa_result.groups)} ({_ext_g} IdP-synced, {len(_pa_result.groups)-_ext_g} Databricks-managed)")
    print(f"  Workspace roles:   {len(_pa_result.workspace_roles)}")
    print(f"  UC permissions:    {len(_pa_result.permissions)}")
    if _pa_result.dead_end_groups:
        print(f"  Dead-end groups (no workspace access): {len(_pa_result.dead_end_groups)}")
        for _dg in _pa_result.dead_end_groups:
            print(f"    - {_dg}")

    # Optional escalation check
    if ESCALATION_CHECK:
        _pa_result.escalation_findings = detect_escalations(_pa_result)
        print(f"\n  Escalation findings: {len(_pa_result.escalation_findings)}")

    df_pa_groups           = result_builder.build_principal_groups_df(_pa_result.groups)
    df_pa_ws               = result_builder.build_principal_ws_df(_pa_result.workspace_roles)
    df_pa_perms            = result_builder.build_principal_perms_df(_pa_result.permissions)
    df_pa_workspace_objects = result_builder.build_workspace_objects_df(_pa_result.workspace_object_grants)

    print("\n✅ Principal audit complete")
    print("   df_pa_groups, df_pa_ws, df_pa_perms")
    if SCAN_WORKSPACE_OBJECTS:
        print("   df_pa_workspace_objects")

In [ ]:
if 'df_pa_groups' in dir() and PRINCIPAL:
    print("\U0001f4c2 Group Memberships  (source: external = IdP-synced, internal = Databricks-managed)")
    display(df_pa_groups)

if 'df_pa_ws' in dir() and PRINCIPAL:
    print("\U0001f4bc Workspace Roles")
    display(df_pa_ws)

if 'df_pa_perms' in dir() and PRINCIPAL:
    print("\U0001f512 Unity Catalog Permissions")
    display(df_pa_perms)


if 'df_pa_workspace_objects' in dir() and PRINCIPAL and SCAN_WORKSPACE_OBJECTS:
    if df_pa_workspace_objects.count() > 0:
        print("🔧 Workspace Object Permissions")
        display(df_pa_workspace_objects)
    else:
        print("ℹ️  No workspace object grants found")

---
## Security Checks

The cells below are independent of the audit mode above.
Enable the relevant widgets and run the cell you need.


In [ ]:
# ============================================================================
# STALE GRANT DETECTION  (requires stale_days > 0 and sql_warehouse_id)
# ============================================================================
_stale_findings = []
if STALE_DAYS and SQL_WAREHOUSE_ID and 'TARGET_GROUP' in dir() and TARGET_GROUP and '_cat_grants' in dir():
    _sql_ws = SQL_WORKSPACE_URL or (_workspaces[0].workspace_url if _workspaces else "")
    if _sql_ws:
        print(f"Checking stale grants (>{STALE_DAYS} days inactive) ...")
        _stale_checker = StaleGrantChecker(client, _sql_ws, SQL_WAREHOUSE_ID, stale_days=STALE_DAYS)
        _stale_findings = _stale_checker.check_catalog_grants(
            _cat_grants, _workspaces[0].workspace_name if _workspaces else "", _sql_ws,
        )
        df_stale = result_builder.build_stale_df(_stale_findings)
        print(f"Found {len(_stale_findings)} stale grant(s)")
        display(df_stale)
    else:
        print("\u26a0\ufe0f  Set sql_workspace_url (or discover workspaces first) to run stale check")
elif STALE_DAYS and not SQL_WAREHOUSE_ID:
    print("\u26a0\ufe0f  stale_days set but sql_warehouse_id is empty - stale check skipped")
else:
    print("\u23e9  Stale check skipped (set stale_days and sql_warehouse_id to enable)")


In [ ]:
# ============================================================================
# PRIVILEGE ESCALATION CHECK  (principal audit only)
# ============================================================================
if '_pa_result' in dir() and PRINCIPAL:
    _esc = _pa_result.escalation_findings if _pa_result.escalation_findings else detect_escalations(_pa_result)
    df_escalation = result_builder.build_escalation_df(_esc)
    if _esc:
        print(f"\u26a0\ufe0f  Escalation risks found: {len(_esc)}")
        display(df_escalation)
    else:
        print("\u2705 No escalation risks (ALL_PRIVILEGES / MANAGE not found)")
else:
    print("\u23e9  Escalation check skipped (run principal audit first)")


In [ ]:
# ============================================================================
# WORKSPACE-LOCAL GROUP DETECTION  (both modes)
# ============================================================================
if CHECK_LOCAL_GROUPS:
    _wss = _workspaces if 'TARGET_GROUP' in dir() and TARGET_GROUP and '_workspaces' in dir() \
           else (_pa_workspaces if '_pa_workspaces' in dir() else [])
    if _wss:
        print(f"Scanning {len(_wss)} workspace(s) for local groups ...")
        _local_checker = LocalGroupChecker(client)
        _local_findings = _local_checker.check_all_workspaces(_wss)
        df_local_groups = result_builder.build_local_groups_df(_local_findings)
        print(f"Found {len(_local_findings)} workspace-local group(s)")
        display(df_local_groups)
    else:
        print("\u26a0\ufe0f  No workspaces available - run group or principal audit first")
else:
    print("\u23e9  Local group check skipped (enable check_local_groups widget)")


---
## Exports

CSV for Excel/SIEM, JSON snapshots for compliance evidence, and optional Delta export.


In [ ]:
# ============================================================================
# CSV EXPORT
# ============================================================================
if TARGET_GROUP and '_cat_grants' in dir():
    print("\U0001f4c4 Group Audit - CSV output:\n")
    _csv_buf = io.StringIO()
    write_group_audit_csv(_cat_grants, _sch_grants, _tbl_grants, _redundancy,
                       workspace_object_grants=_ws_obj_grants if '_ws_obj_grants' in dir() else None,
                       output=_csv_buf)
    print(_csv_buf.getvalue()[:3000], "..." if len(_csv_buf.getvalue()) > 3000 else "")
    # To save: open("/dbfs/tmp/group_audit.csv","w").write(_csv_buf.getvalue())

elif PRINCIPAL and '_pa_result' in dir():
    print("\U0001f4c4 Principal Audit - CSV output:\n")
    _csv_buf = io.StringIO()
    write_principal_audit_csv(_pa_result, _pa_result.escalation_findings, output=_csv_buf)
    print(_csv_buf.getvalue()[:3000], "..." if len(_csv_buf.getvalue()) > 3000 else "")
    # To save: open("/dbfs/tmp/principal_audit.csv","w").write(_csv_buf.getvalue())

else:
    print("\u23e9  Run group or principal audit first, then re-run this cell.")


In [ ]:
# ============================================================================
# SNAPSHOT SAVE + DIFF (SOC 2 / ISO 27001 evidence)
# ============================================================================
if TARGET_GROUP and '_cat_grants' in dir():
    _snap = build_group_snapshot(TARGET_GROUP, _members, _cat_grants, _sch_grants, _tbl_grants,
                              workspace_object_grants=_ws_obj_grants if '_ws_obj_grants' in dir() else None)
elif PRINCIPAL and '_pa_result' in dir():
    _snap = build_principal_snapshot(_pa_result)
else:
    _snap = None
    print("\u23e9  Run an audit first, then re-run this cell.")

if _snap:
    if SAVE_SNAPSHOT:
        save_snapshot(_snap, SAVE_SNAPSHOT)
        print(f"\u2705 Snapshot saved to {SAVE_SNAPSHOT}")

    if BASELINE_SNAPSHOT:
        try:
            _baseline = load_snapshot(BASELINE_SNAPSHOT)
            _diff = diff_snapshots(_baseline, _snap)
            if _diff.has_changes:
                print(f"\u26a0\ufe0f  Changes detected vs baseline ({_baseline.get('timestamp','?')}):")
                print(f"   Grants added:    {len(_diff.grants_added)}")
                print(f"   Grants removed:  {len(_diff.grants_removed)}")
                print(f"   Members added:   {len(_diff.members_added)}")
                print(f"   Members removed: {len(_diff.members_removed)}")

                for g in _diff.grants_added:
                    print(f"   + [{g.get('securable_type','')}] {g.get('securable_name','')} - "
                          f"{g.get('principal',g.get('via_group',''))} ({', '.join(g.get('privileges',[]))})")
                for g in _diff.grants_removed:
                    print(f"   - [{g.get('securable_type','')}] {g.get('securable_name','')} - "
                          f"{g.get('principal',g.get('via_group',''))} ({', '.join(g.get('privileges',[]))})")
                for m in _diff.members_added:
                    print(f"   + member: {m.get('display_name',m.get('group_name','?'))} ({m.get('type','?')})")
                for m in _diff.members_removed:
                    print(f"   - member: {m.get('display_name',m.get('group_name','?'))} ({m.get('type','?')})")
            else:
                print("\u2705 No changes detected vs baseline snapshot")
        except FileNotFoundError:
            print(f"\u26a0\ufe0f  Baseline not found: {BASELINE_SNAPSHOT}")
    elif not SAVE_SNAPSHOT:
        print("\u2139\ufe0f  Set save_snapshot and/or baseline_snapshot widgets to use this feature")


In [ ]:
# ============================================================================
# DELTA EXPORT (optional - set export_delta_path widget to enable)
# ============================================================================
if EXPORT_DELTA_PATH:
    from pyspark.sql import functions as F
    _ts = F.lit(datetime.now().isoformat())
    _tgt = TARGET_GROUP or PRINCIPAL or "unknown"

    _exports = {}
    if TARGET_GROUP and '_cat_grants' in dir():
        _exports = {
            "permissions":   df_grants,
            "membership":    df_membership,
            "redundancy":    df_redundancy,
        }
        if 'df_schema_grants' in dir(): _exports["schema_grants"] = df_schema_grants
        if 'df_table_grants'  in dir(): _exports["table_grants"]  = df_table_grants
    elif PRINCIPAL and '_pa_result' in dir():
        _exports = {
            "pa_groups":  df_pa_groups,
            "pa_ws_roles":df_pa_ws,
            "pa_perms":   df_pa_perms,
        }

    for _name, _df in _exports.items():
        _path = f"{EXPORT_DELTA_PATH}/{_name}"
        (_df.withColumn("audit_timestamp", _ts)
           .withColumn("target", F.lit(_tgt))
           .write.format("delta").mode("append").option("mergeSchema","true").save(_path))
        print(f"  \u2705 {_name} -> {_path}")
    print(f"\u2705 Exported {len(_exports)} table(s) to {EXPORT_DELTA_PATH}")
else:
    print("\u23e9  Delta export disabled (set export_delta_path widget to enable)")
